In [1]:
import os
import sys
import numpy as np
from parameters import l_samples,s_1k,s_2k,s_3k,s_4k,b_4k

MCELL_PATH = os.environ.get('MCELL_PATH', '')
sys.path.append(os.path.join(MCELL_PATH, 'lib'))

import mcell as m

print("Import of MCell was sucessful")

every_n = 100000

bngl_dir = 'bngl_files/'


Import of MCell was sucessful


In [4]:
#write functions which automatically check for a specific BNGL file and otherwise write them

def write_samplel_bngl(l,N): 
    #define the different l distributions here
    file_path = bngl_dir+'example_'+l+'.bngl'
    if os.path.exists(file_path):
        return file_path
    else:
        exp_file = open(bngl_dir+'example_.bngl', 'r')
        new_file = open(filepath, 'w')
        if l[0] == '2': 
            l_samples = np.array([l[1],l[2]], dtype = int)
            partitions = np.array([0,1], dtype = int)
        if l[0] == 'dist': 
            partitions = np.zeros([l_samples.shape[0]], dtype = int)
            partitions[s_2k] = 1
            partitions[s_3k] = 2
            partitions[s_4k] = 3
            partitions[b_4k] = 4
        
        for line in exp_file: 
            if not '%%%%insert' in line: 
                #need to check whether line break is already in it. 
                new_file.write(line)
            elif '%%%%insert numbers' in line: 
                for l_i in l_samples:
                    new_file.write('    nucl_'+l_i+' '+str(N)+'/n')
                
            elif '%%%%insert nucl def' in line: 
                if l[0] == '2': 
                    new_file.write('    nucl(l,r,R,i~0~1,t~0~1,s~0~1)/n')
                elif l[0] == 'dist':
                    new_file.write('    nucl(l,r,R,i~0~1,t~0~1,s~0~1~2~3~4)/n')
            
            elif '%%%%insert mRNA def' in line:
                for l_ind, l_i in enumerate(l_samples): 
                    mid = ''
                    ind = partitions[l_ind]
                    for x in range(l_i): 
                        mid = mid+'.nucl(R,t~0,i~0,s~'+str(ind)+',l!'+str(x)+',r!'+str(x+1)+')'
                    new_file.write('@CP:nucl(R,t~0,i~1,s~'+str(ind)+',r!0,l)'+mid+'.nucl(R,t~1,i~0,s~'+str(ind)+',l!'+str(x+1)+',r) nucl_'+str(l_i)+'/n')
            
            elif '%%%%insert rection equation' in line:
                if l[0] =='2':
                    new_file.write('nucl(R!1, t~1, s~0).Ribo(N!1)-> P(s~0)+ nucl(R, t~1, s~0)+ Ribo(N) term_r /n')
                    new_file.write('nucl(R!1, t~1, s~1).Ribo(N!1)-> P(s~1)+ nucl(R, t~1, s~1)+ Ribo(N) term_r /n')
                if l[0] =='dist':
                    for i in range(5): 
                        new_file.write('nucl(R!1, t~1, s~'+str(i)+').Ribo(N!1)-> P(s~'+str(i)+')+ nucl(R, t~1, s~'+str(i)+')+ Ribo(N) term_r /n')
                
            elif '%%%%insert observ' in line:
                if l[0] =='2': 
                    new_file.write('Molecules p_s P(s~0)/n')
                    new_file.write('Molecules r_initiated_s nucl(R!1, i~1, s~0).Ribo(N!1)/n')
                    new_file.write('Molecules r_elongating_s nucl(R!1, t~0, i~0, s~0).Ribo(N!1)/n')
                    new_file.write('Molecules r_terminating_s nucl(R!1, t~1, i~0, s~0).Ribo(N!1)/n')
                    
                    new_file.write('Molecules p_l P(s~1)/n')
                    new_file.write('Molecules r_initiated_l nucl(R!1, i~1, s~1).Ribo(N!1)/n')
                    new_file.write('Molecules r_elongating_l nucl(R!1, t~0, i~0, s~1).Ribo(N!1)/n')
                    new_file.write('Molecules r_terminating_l nucl(R!1, t~1, i~0, s~1).Ribo(N!1)/n')
                    
                if l[0] =='dist':
                    for i in range(5):
                        new_file.write('Molecules p_'+str(i)+' P(s~'+str(i)+')/n')
                        new_file.write('Molecules r_initiated_'+str(i)+' nucl(R!1, i~1, s~'+str(i)+').Ribo(N!1)/n')
                        new_file.write('Molecules r_elongating_'+str(i)+' nucl(R!1, t~0, i~0, s~'+str(i)+').Ribo(N!1)/n')
                        new_file.write('Molecules r_terminating_'+str(i)+' nucl(R!1, t~1, i~0, s~'+str(i)+').Ribo(N!1)/n')
                
        exp_file.close()
        new_file.close()
        return file_path
    
    
    
    

def check_and_write_bngl(R, Y, l):
    #the different distributions of mRNAs are predefined in the example file 
    #doing this automatically would mean a lot more inserting than just exchanging numbers
    #we will abuse chatGPT as a finite state machine to get those without too much typing
    file_path = bngl_dir+str(R)+'ribos_'+str(Y)+'aminos_'+l+'.bngl'
    if os.path.exists(file_path):
        return file_path
    else:
        exp_file = open(bngl_dir+'example_'+l+'.bngl', 'r')
        new_file = open(filepath, 'w')
        
        for line in exp_file: 
            if not '%%%%insert' in line: 
                #need to check whether line break is already in it. 
                new_file.write(line)
            elif '%%%%insert ribo' in line: 
                new_file.write('    Ribo_i '+str(R)+'/n')
                
            elif '%%%%insert Y' in line: 
                new_file.write('    BkP_i '+str(Y)+'/n')
        
        
        exp_file.close()
        new_file.close()
        return file_path
    

SyntaxError: invalid syntax (8321745.py, line 11)

In [ ]:

viz_output = m.VizOutput(
    output_files_prefix = './viz_data/seed_00002/Scene',
    every_n_timesteps= every_n
    )

# Creting geometry
#vol_cp = 0.50588 # units of um3
#r = (3*vol_cp/(4*np.pi))**(1/3.0) # units of microns

#cp = m.geometry_utils.create_icosphere(
#    'CP', radius = r, subdivisions=2)
#cp.is_bngl_compartment = True
#cp.surface_compartment_name = 'PM'

model = m.Model()

model.add_viz_output(viz_output)
#model.add_geometry_object(cp)


#load the model file 
model.load_bngl(
    'model_no_amino.bngl', 
    observables_path_or_file='rib_dat_no_amino2.gdat')

#open bngl file and load the parameters into a dictionary
param_dict = m.bngl_utils.load_bngl_parameters('model_no_amino.bngl')
#ITERATIONS = param_dict['ITERATIONS']

# Specifies periodicity of visualization output
for count in model.counts:
    count.every_n_timesteps = every_n

#Do not use bng units:
model.config.use_bng_units = False

# Check to see if total iterations is defined as a global parameter
if 'ITERATIONS' not in globals():
    ITERATIONS = 1e6 #1e6

# Total_Iterations if not defined explicitly default to 1e6
model.config.total_iterations = ITERATIONS
model.config.time_step = 1e-6  #default is 1e-6

model.config.partition_dimension = 1.5 # 1.5 was before


#maybe i need to specify the species as a bngl species as well

#by CellBlender the following part is added to a subsystem, we will try to add it to the model directly
#we try this so we can set variables rates
#var_rates = np.loadtxt('ca_rates.txt')
#var_rate_Bk_to_Ca_0 = var_rates[:,[0,1]].tolist()
#var_rate_Ca_to_Bk_1 = var_rates[:,[0,2]].tolist()#

#Bk_to_Ca = m.ReactionRule(
#    name = 'Bk_to_Ca',
#    reactants = [ m.Complex('Bk') ],
#    products = [ m.Complex('Ca') ],
#    variable_rate = var_rate_Bk_to_Ca_0
#)

#Ca_to_Bk = m.ReactionRule(
#    name = 'Ca_to_Bk',
#    reactants = [ m.Complex('Ca') ],
#    products = [ m.Complex('Bk') ],
#    variable_rate = var_rate_Ca_to_Bk_1
#)


#model.add_reaction_rule(Bk_to_Ca)
#model.add_reaction_rule(Ca_to_Bk)
######end insert from the CellBlender code

model.initialize()

model.export_viz_data_model()
#model.export_data_model()

model.run_iterations(ITERATIONS)

model.end_simulation()